# Typology rhythm automation tutorial by Yan

## Made for Ling 199



*   Always thank your fellow linguists for completing the most important task, the transcription!
* (IN PRAAT) [OPTIONAL] What you can do is go to Text writing settings. In the Settings submenu of the Praat menu, and there set the output encoding to UTF-8. Praat will from then on save your text files in the UTF-8 format, which means one byte for every ASCII character and 2 to 4 bytes for every non-ASCII character. Especially on Linux, many programs understand UTF-8 text and will display the correct characters. Programs such as SPSS do not understand UTF-8 but will still display ASCII characters correctly; for instance, the names München and Wałęsa may appear as MÃ÷nchen and WaÅ,Ä™sa or so.
*   Make sure sheet does not have empty cells in obvious locations such as phoneme or word column.
*   For single-parsed sentences, remove empty rows before running the script.
*   Always check for any weird inconsistency to save yourself time.
* Report any new language nuances to Sherry, and consult with Yan on how to fix.



Clone the script to load it in for use, you should see it in the left menu after navigating to the folder with `cd`.



In [ ]:
!git clone https://github.com/YLashchev/IPA_parser.git

### Local usage (non-Colab)
If you already have this repo locally, skip the clone step and run the cell below instead.


In [ ]:
%cd ..
!.venv/bin/python -m pip install -e .
!.venv/bin/python -m ipykernel install --user --name ipa-parser

In [ ]:
!.venv/bin/python -c "import sys; print(sys.executable)"
!.venv/bin/python -c "import ipa; print(ipa.__file__)"

In [ ]:
# Path no longer needed with editable install from repo root

In [ ]:
# For Colab: change to IPA_parser directory after cloning
# %cd IPA_parser
# !pip install -e .
# (Local usage: skip this cell - use parent directory install)

### Import custom classes from scripts and Pandas to work with DataFrame structure.

In [ ]:
from ipa import CustomCharacter, IPA_CHAR, IPAString
from ipa.pipeline import build_final_dataframe, load_excel, configure_custom_characters
from ipa.config import load_language_config
import pandas as pd
import numpy as np 

# Pre-processing

## Configure language settings



*  Set `GEMINATE` value to True/False if present, or define custom geminates if not.
*   clear previously defined custom characters if any (to be safe).
*   Add custom characters using `CustomCharacter` class for all affricates/diphtongs. Otherwise count will be wrong.
*   For now set affricate to 'CONSONANT', Diphtong to 'VOWEL' with rank = 1.





In [ ]:
GEMINATE = False
CustomCharacter.clear_all_chars()
CustomCharacter.add_char('OP', 'PAUSE', rank = 0)
CustomCharacter.add_char('SP', 'PAUSE', rank = 0)
CustomCharacter.add_char('ʁ', 'CONSONANT', rank = 1)
CustomCharacter.add_char('t͡s', 'CONSONANT', rank = 1)












#ZwaraBerber

In [ ]:
GEMINATE = True


CustomCharacter.clear_all_chars()



CustomCharacter.add_char('OP', 'PAUSE', rank = 0)
CustomCharacter.add_char('SP', 'PAUSE', rank = 0)
CustomCharacter.add_char('sˁ', 'CONSONANT',rank=1)
CustomCharacter.add_char('nˁ', 'CONSONANT',rank=1)
CustomCharacter.add_char('dˁ', 'CONSONANT',rank=1)
CustomCharacter.add_char('lˁ', 'CONSONANT',rank=1)
CustomCharacter.add_char('mˁ', 'CONSONANT',rank=1)
CustomCharacter.add_char('rˁ', 'CONSONANT',rank=1)
CustomCharacter.add_char('nˁnˁ', 'CONSONANT',rank=1)
CustomCharacter.add_char('mˁmˁ', 'CONSONANT',rank=1)


# Northwest Sahaptin

In [ ]:
GEMINATE = False

CustomCharacter.clear_all_chars()
CustomCharacter.add_char('OP', 'PAUSE', rank = 0)
CustomCharacter.add_char('SP', 'PAUSE', rank = 0)
CustomCharacter.add_char('t͡s', 'CONSONANT',rank=1)
CustomCharacter.add_char('t͡s’', 'CONSONANT',rank=1)
CustomCharacter.add_char('t͡ʃ’', 'CONSONANT',rank=1)
CustomCharacter.add_char('t͡ʃ', 'CONSONANT',rank=1)
CustomCharacter.add_char('q’', 'CONSONANT',rank=1)



# Oriental Hebrew

In [ ]:
GEMINATE = False

CustomCharacter.add_char('OP', 'PAUSE', rank = 0)
CustomCharacter.add_char('SP', 'PAUSE', rank = 0)
CustomCharacter.add_char('t͡s', 'CONSONANT',rank=1)
CustomCharacter.add_char('ts', 'CONSONANT',rank=1)



In [ ]:
GEMINATE = False
CustomCharacter.add_char('ᵐ̥p', 'CONSONANT',rank=1)
CustomCharacter.add_char('ᵐ', 'CONSONANT',rank=0)
CustomCharacter.add_char('ᵐ̥', 'CONSONANT',rank=0)
CustomCharacter.add_char('ʈ͡ʂ', 'CONSONANT',rank=1)
CustomCharacter.add_char('ɖ͡ʐ', 'CONSONANT',rank=1)
CustomCharacter.add_char('t͡sʲ', 'CONSONANT',rank=1)
CustomCharacter.add_char('ʈ̬͡ʂ̬', 'CONSONANT',rank=1)

#ʈ͡ʂ
#ʈ̬͡ʂ̬
#ɖ͡ʐ
#t͡sʲ

#ᵐ̥p
#ᵐ

## Load the data in

* You can drag the excel sheet into the menu in the right then (right-click copy path) and enter it into the input.
* We then fix the original columns names to be safe.
* Copy path from sheets to use as example.

In [ ]:
#file_path = input("Enter the path to your Excel file: ")
#/content/IPA_parser/data/sheets/NorthernTepehuan.xlsx
#/content/IPA_parser/data/sheets/NorthwestSahaptin.xlsx



# Read the Excel file using the provided path
df = pd.read_excel('/Users/yanlashchev/Desktop/IPA_Project/IPA_parser/data/sheets/Sheets/NorthwestSahaptin.xlsx', engine='openpyxl')

# Correct original columns names to be safe
new_columns = ['Filename', 'Sentence', 'Word', 'Phoneme', 'Begin', 'End', 'Duration (ms.)']
df.columns = new_columns

## Pipeline Execution

Run the complete IPA processing pipeline using `build_final_dataframe` from `ipa.pipeline`.

This function handles all processing steps:
1. Insert sentence pause markers (SP)
2. Assign pause labels (OP, SP)
3. Build word list and detect mismatches
4. Compute word-level metrics (length, syllable count, duration)
5. Compute syllable-level metrics (length, duration, coda, stress)
6. Annotate segments (type, coda complexity, stress)
7. Compute sentence metrics
8. Compute ISI (inter-stress interval) metrics
9. Fill pause columns and remove pause rows
10. Format and select final columns

Returns: `(final_df, mismatches)` tuple

In [ ]:
# Run the complete IPA processing pipeline
final_df, mismatches = build_final_dataframe(df, geminate=GEMINATE)

# Display any phoneme-alignment mismatches
if mismatches:
    print(f"Found {len(mismatches)} phoneme-alignment mismatches:")
    for row_num, word in mismatches:
        print(f"  Row {row_num}: {word}")
else:
    print("No phoneme-alignment mismatches detected.")

print(f"\nProcessed {len(final_df)} phoneme rows")
print(f"Shape: {final_df.shape}")
print(f"\nColumns: {list(final_df.columns)}")

In [ ]:
# Display the processed dataframe
final_df.head(20)

### Helper functions `inset_sp(df)` and `assign_pauses`

`insert_sp(df)` does a couple things:

1.   Adds 'SP' to Sentence column (for verification purposes and my sanity)
2.   If there is no empty cell between sentences but there is a transition add a row. (This is for single-sentence parsed transcriptions). This is why removing the empty rows is important pre-processing. We do not want to mistake a NaN value for an empty row. Luckily the data is always short enough to visibly check.
3. For the SP added to the single-sentence parsed transcriptions we put duration of 0, and handle this case appropriately below when neccessary.

`assign_pauses`
* puts 'OP' and 'SP' into the respective word and phoneme columns for sanity check.



### Word Level

* Get original word list by first appearace (i.e., non-repeat) and index of occurence.

* Logic is simple, if next word is the same as previous word still the same word.

***NOTE***: This will be an issue if there is a language where the word is repeated twice on purpose. "The food was **so so**"




`find_mismatches_with_phoneme_alignment`
* loops through each word and compare the concatenated phoneme with the word itself. If there is a mismatch, we add the word index and the word itself to a list. We return the list of mismatches.
* Sometimes mismatch is due to diff characters with same representation like small cap latin letters which just requires a simple replacement.


#### Helper function and Word Columns


*   `collapse_nested_list` does what it says, takes a nested list of lists and makes it one vector.
*   `word_columns`

  1.  Iterates unique words getting phonological length based on custom script
  2.  repeats this word by the phonological length alongside the **Word Number** in sentence.
  3. reset at occurence of SP.
  4. returns the repeated word list, the word labels (Word Number column), and the unique index for every word.


* `process_durations`
  * gets duration based on unique index list.


checking lengths match.

`Word_Length_By_Phoneme`


*   Creats WordLengthByPhoneme Column by using `total_length()` from `IPAString`.
*   Appends OP and SP as they were.

`Word_Length_By_Syllable`


*   Creats WordLengthBySyllable Column by using `syllables` from `IPAString`.
*   Appends OP and SP as they were.




## Partial Sheet Check

I line up the original word data with the repeated words based on phonological length. I check the sheet 'check.xlsx' in the menu and see where the misalignment happens, and then either fix the missing character or add the custom character to the list above.

### Rinse & Refresh.

In [ ]:
#Keep columns
df['WordLengthByPhoneme'] = wlbp
df['WordLengthBySyllable'] = wlbs
df['WordNumber'] = word_labels

#temporary
df['unique_word_idx'] = unique_word_idx

print(df)

#use temporary for durations
summed_word_durations = process_durations(df,'unique_word_idx')
df['WordDuration'] = summed_word_durations

### Syllable Level

`pre_syllable_columns`
* gets the list of syllables for each word using `.syllables` from `IPAString`.


`syllable_number`
* creates the SyllableNumber column by using the nested list and IPAString methods as before.

Create syllable lists, SyllableNumber column, and placeholder columns

**SyllableLengthByPhoneme Column**
* use `.total_length()` from IPAString class on temporary syllable column

**SyllableLengthByPhoneme & SyllableDuration columns**

### Sentence level

`SentenceDUration`
1. We loop through every row in the dataframe, and get the index number of the word (e.g. 1, 2, 3, etc.). We also get the duration of the word.
2. If the index number is "OP" or "SP", we directly append the duration to the output list.
3. If the index number is not "OP" or "SP", we need to sum up the durations of the words in the same group. To do that, we need to keep track of the current word, the sum of the durations of the current group, and the count of the words in the current group.
4. If the current word is the same as the previous word, we add the duration to the sum, and increment the count by 1.
5. If the current word is not the same as the previous word, we append the sum multiplied by the count to the output list, and reset the current word, the sum, and the count.
6. After we finish looping through all the rows in the dataframe, we need to add the sum multiplied by the count for the last word to the output list.


`by_sentence_count`

1. We split the list into sublists whenever 'SP' is encountered
2. We iterate over each sublist and calculate the length of the sublist
3. We add the length of each sublist to the result list the number of times equal to the length of the sublist
4. We append 'SP' after each complete sentence except the last one
5. We return two lists: a list of phoneme counts and a list of syllable counts for each sentence.

### SegmentType Column
*   use `char_only()` to remove diacritics and special symbols and `segment_type` to get the type of character.
* We can use the original phoneme list



### CodaComplexity Column

*   Use `.coda` from `IPAString` on temporary syllables column to return coda for each syllable




In [ ]:
coda_column = coda_column(repeated_syllable_list)
print('coda_column',len(coda_column))

#fix this to actually count the coda
df['CodaComplexity'] = coda_column

### Stress Column


*   Use `.stress()` from `IPAString` on syllable column to return type of stress. (primary/secondary/unstressed)




###Interstress Intervals

`create_isi_blocks`
1. Iterate through each row in the DataFrame, starting from the second row.
2. Check if the current syllable is stressed. If so, set the encountered_stressed_syllable to True.
3. Check if the current syllable is stressed and there was a change in unique_syll_idx.
   Only do this if we have previously encountered a stressed syllable.
4. If the above condition is met, set the new_block column to True.
5. Reset the block if 'SP' is detected with duration 0 and set the next syllable to start a new block if it's stressed.
6. Set the first entry to start a new block only if it's stressed.

`get_isi_index`

*   gets the index location of every interstress interval start

`calculate_interstress_duration(df, isi_idx, summed_col_name='', excluded_words=None)`

* Gets the duration of the ISI block. 'summed_col_name is' the parameter for whichever column you are creating. If you need ISI with pause or without. For without you have an additional parameter excluded_words which is used for OP/SP.


`ISI_Pause` checks for a difference between pause columns (a difference would imply a pause) it then further checks if the pause occurs within an SP and then labels them accordingly

`ISI_By_Segment` and `ISI_By_Syllable` uses similar logic as before to get the repeated values. By_Segment returns 3 values, segment (C+V), consonant, vowel

`fill_pause_columns` creates OtherPauses & SentencePauses columns.

`remove_pause_rows` just removes those rows for final sheet.

`update_isi_columns` sets all columns that concern ISI to 'N/A' when there is no stress encountered whatsoever.

The rest is just finalizing the sheet how Ryan requires it.

In [ ]:
print("OP:", (df['Word'] == 'OP').sum(), "SP:", (df['Word'] == 'SP').sum())
print(df[['Word','Duration (ms.)']].head(20))

Let output.xlsx load into folder menu.

In [ ]:
final_df.to_excel("/Users/yanlashchev/Desktop/IPA_Project/IPA_parser/output.xlsx", index=False, na_rep="N/A")

# FIN